# 🔧 Pipeline ML — Preprocessing pour Modèles Immobilier

Ce notebook exécute le pipeline de prétraitement étape par étape avec des visualisations avant/après chaque transformation.

**Pipeline:** `src/cleaning/ml_pipeline.py`  
**Output:** `data/final/` → X_train, X_test, y_train, y_test

---

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import glob
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

try:
    import plotly.express as px
    PLOTLY = True
except ImportError:
    PLOTLY = False
    print('plotly non disponible — uniquement matplotlib')

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print('✅ Setup OK')

## 1. Charger les données brutes

In [ ]:
# Charger le fichier combiné le plus récent
files = glob.glob(os.path.join('..', 'data', 'processed', 'immobilier_maroc_*.csv'))
latest = max(files, key=os.path.getctime)
df_raw = pd.read_csv(latest)

print(f'📄 Fichier : {latest}')
print(f'📊 Shape   : {df_raw.shape}')
print(f'📋 Colonnes: {list(df_raw.columns)}')
df_raw.head(5)

In [ ]:
# Valeurs manquantes initiales
missing = df_raw.isnull().sum()
missing = missing[missing > 0]
print('Valeurs manquantes avant pipeline:')
display(pd.DataFrame({'Manquantes': missing, '%': (missing/len(df_raw)*100).round(1)}))

## 2. Lancer le Pipeline Complet

In [ ]:
import importlib.util, pathlib

# Import dynamique du pipeline
spec = importlib.util.spec_from_file_location(
    'ml_pipeline',
    str(pathlib.Path('..') / 'src' / 'cleaning' / 'ml_pipeline.py')
)
ml_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ml_module)

# Exécuter
pipeline = ml_module.MLPipeline(
    data_dir   = os.path.join('..', 'data', 'processed'),
    output_dir = os.path.join('..', 'data', 'final')
)
pipeline.run()
print('\n✅ Pipeline terminé!')

## 3. Rapport du Pipeline

In [ ]:
with open(os.path.join('..', 'data', 'final', 'pipeline_report.json'), encoding='utf-8') as f:
    report = json.load(f)

print(f"Status  : {report['status']}")
print(f"Timestamp: {report['timestamp']}")
print()

# Résumé des étapes
rows_data = []
for step, info in report['steps'].items():
    if 'rows_before' in info:
        rows_data.append({
            'Étape': step,
            'Avant': info['rows_before'],
            'Après': info['rows_after'],
            'Supprimées': info['dropped']
        })

df_report = pd.DataFrame(rows_data)
display(df_report)

In [ ]:
# Visualisation de la réduction de données à chaque étape
fig, ax = plt.subplots(figsize=(12, 5))
steps = df_report['Étape'].str.replace('step\\d+_', '', regex=True).str.upper()
bars = ax.bar(steps, df_report['Après'], color=sns.color_palette('Blues_d', len(steps)))
ax.set_title('Lignes restantes après chaque étape du pipeline', fontsize=15, fontweight='bold')
ax.set_ylabel('Nombre de lignes')
ax.set_ylim(0, df_report['Avant'].max() * 1.15)
for bar, val in zip(bars, df_report['Après']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(val),
            ha='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Charger les Datasets Finaux

In [ ]:
final_dir = os.path.join('..', 'data', 'final')

X_train = pd.read_csv(f'{final_dir}/X_train.csv')
X_test  = pd.read_csv(f'{final_dir}/X_test.csv')
y_train = pd.read_csv(f'{final_dir}/y_train.csv')
y_test  = pd.read_csv(f'{final_dir}/y_test.csv')

print('📦 Datasets chargés:')
print(f'  X_train : {X_train.shape}')
print(f'  X_test  : {X_test.shape}')
print(f'  y_train : {y_train.shape}')
print(f'  y_test  : {y_test.shape}')
print(f'\n  Features: {list(X_train.columns)}')

# Vérification : 0 NaN
nan_train = X_train.isnull().sum().sum()
nan_test  = X_test.isnull().sum().sum()
print(f'\n✅ NaN dans X_train : {nan_train}')
print(f'✅ NaN dans X_test  : {nan_test}')

In [ ]:
X_train.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

## 5. Distributions Avant / Après Pipeline

In [ ]:
# Refaire le raw pour comparer
df_raw_num = df_raw[['prix', 'surface_m2']].copy()
df_raw_num['prix']       = pd.to_numeric(df_raw_num['prix'], errors='coerce')
df_raw_num['surface_m2'] = pd.to_numeric(df_raw_num['surface_m2'], errors='coerce')

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for i, col in enumerate(['prix', 'surface_m2']):
    # Avant
    data_before = df_raw_num[col].dropna()
    axes[i][0].hist(data_before, bins=30, color='#e74c3c', edgecolor='white', alpha=0.8)
    axes[i][0].set_title(f'{col.upper()} — AVANT pipeline', fontsize=13, fontweight='bold')
    axes[i][0].axvline(data_before.mean(), color='black', linestyle='--', label=f'μ={data_before.mean():,.0f}')
    axes[i][0].legend()

    # Après (from X_train/X_test)
    if col in X_train.columns:
        # Scaled — just show distribution shape
        data_after = pd.concat([X_train[col], X_test[col]])
        axes[i][1].hist(data_after, bins=30, color='#2ecc71', edgecolor='white', alpha=0.8)
        axes[i][1].set_title(f'{col.upper()} — APRÈS pipeline (scaled)', fontsize=13, fontweight='bold')
        axes[i][1].axvline(data_after.mean(), color='black', linestyle='--', label=f'μ={data_after.mean():.2f}')
        axes[i][1].legend()

plt.suptitle('Distributions Avant / Après Pipeline', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Distribution de la Target (prix)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

all_y = pd.concat([y_train, y_test])['prix']

# Distribution normale
axes[0].hist(all_y, bins=25, color='#3498db', edgecolor='white', alpha=0.85)
axes[0].axvline(all_y.mean(),   color='red',   linestyle='--', linewidth=2, label=f'Moyenne: {all_y.mean():,.0f} MAD')
axes[0].axvline(all_y.median(), color='green', linestyle='--', linewidth=2, label=f'Médiane: {all_y.median():,.0f} MAD')
axes[0].set_title('Distribution du Prix (target)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Prix (MAD)')
axes[0].legend()

# Log-scale
axes[1].hist(np.log1p(all_y), bins=25, color='#9b59b6', edgecolor='white', alpha=0.85)
axes[1].set_title('Distribution du Prix — log(1+prix)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('log(1 + prix)')

plt.tight_layout()
plt.show()

skew = all_y.skew()
print(f'Skewness prix           : {skew:.3f}')
print(f'Skewness log(1+prix)    : {np.log1p(all_y).skew():.3f}')
if abs(skew) > 1:
    print('\n⚠️ Distribution asymétrique — considérer transformation log(prix) pour les modèles linéaires')

## 7. Corrélation des Features

In [ ]:
# Matrice de corrélation complète
df_full = X_train.copy()
df_full['prix'] = y_train['prix'].values

corr = df_full.corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Matrice de Corrélation — Dataset ML Final', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Corrélation avec le target
print('\n📈 Corrélation avec PRIX (target):')
target_corr = corr['prix'].drop('prix').sort_values(ascending=False)
display(target_corr.to_frame().style.background_gradient(cmap='RdYlGn', axis=0))

## 8. Résumé Final

In [ ]:
print('=' * 60)
print('📋 RÉSUMÉ DU DATASET ML')
print('=' * 60)
print(f'  X_train : {X_train.shape[0]:>5} lignes × {X_train.shape[1]} features')
print(f'  X_test  : {X_test.shape[0]:>5} lignes × {X_test.shape[1]} features')
print(f'  y_train : {y_train.shape[0]:>5} lignes (target: prix)')
print(f'  y_test  : {y_test.shape[0]:>5} lignes')
print()
print(f'  NaN dans X_train : {X_train.isnull().sum().sum()}')
print(f'  NaN dans X_test  : {X_test.isnull().sum().sum()}')
print()
print(f'  Prix min (train) : {y_train["prix"].min():>12,.0f} MAD')
print(f'  Prix max (train) : {y_train["prix"].max():>12,.0f} MAD')
print(f'  Prix moyen       : {y_train["prix"].mean():>12,.0f} MAD')
print()
print('  Features :')
for col in X_train.columns:
    print(f'    - {col}')
print('=' * 60)
print('🎯 Dataset prêt pour l\'entraînement des modèles ML!')
print('=' * 60)